In [2]:
# %%
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import importlib

# Locate project root
project_root = Path.cwd().parents[2]

# Add project root to Python's module search path
sys.path.insert(0, str(project_root))

# Import reusable modules
from src.data_loader import load_cv_data
import src.feature_engineering as fe

# Load the latest feature-engineering implementation
importlib.reload(fe)

print("Project root:", project_root)
print("Setup completed.")

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Setup completed.


In [3]:
# %%
fold_datasets = load_cv_data(project_root)

# Convert skill IDs into Python sets
def convert_skill_id(skill_id):
    if pd.isna(skill_id):
        return set()

    return set(str(skill_id).split("_"))


for train_df, val_df in fold_datasets:

    train_df["skill_set"] = train_df["skill_id"].apply(
        convert_skill_id
    )

    val_df["skill_set"] = val_df["skill_id"].apply(
        convert_skill_id
    )


# Display fold sizes
for fold_number, (train_df, val_df) in enumerate(
    fold_datasets, start=1
):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )


# Inspect skill information
fold_train_df, fold_val_df = fold_datasets[0]

print("\nSkill ID examples:")
print(fold_train_df["skill_id"].head(10).tolist())

print("\nConverted skill sets:")
print(fold_train_df["skill_set"].head(10).tolist())

print("\nSkill set data type:")
print(type(fold_train_df["skill_set"].iloc[0]))

print("\nMissing skill IDs:")
print(fold_train_df["skill_id"].isna().sum())

Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)

Skill ID examples:
['2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_48_77', '2_48_79', '2_37_48', '2_48_79']

Converted skill sets:
[{'37', '2', '70'}, {'37', '2', '70'}, {'37', '2', '70'}, {'37', '2', '70'}, {'37', '2', '70'}, {'37', '2', '70'}, {'77', '37', '2', '48'}, {'2', '79', '48'}, {'37', '2', '48'}, {'2', '79', '48'}]

Skill set data type:
<class 'set'>

Missing skill IDs:
11163


In [4]:
# %%
from collections import Counter

fold_train_df, fold_val_df = fold_datasets[0]

# Count how frequently each skill ID appears
skill_counts = Counter(
    skill
    for skill_set in fold_train_df["skill_set"]
    for skill in skill_set
)

total_questions = len(fold_train_df)

print("Total training questions:", total_questions)

print("\nMost frequent skill IDs:")

for skill, count in skill_counts.most_common(15):
    print(
        f"Skill {skill}: {count} questions "
        f"({count / total_questions * 100:.2f}%)"
    )

Total training questions: 177324

Most frequent skill IDs:
Skill 47: 11597 questions (6.54%)
Skill 311: 11090 questions (6.25%)
Skill 277: 7340 questions (4.14%)
Skill 280: 7218 questions (4.07%)
Skill 70: 6074 questions (3.43%)
Skill 79: 5827 questions (3.29%)
Skill 50: 5453 questions (3.08%)
Skill 17: 5148 questions (2.90%)
Skill 77: 4974 questions (2.81%)
Skill 27: 4698 questions (2.65%)
Skill 49: 4590 questions (2.59%)
Skill 18: 4531 questions (2.56%)
Skill 85: 4528 questions (2.55%)
Skill 10: 4451 questions (2.51%)
Skill 279: 4282 questions (2.41%)


In [5]:
# %%
skill_set_sizes = fold_train_df["skill_set"].apply(len)

print("Skill count distribution:")
print(skill_set_sizes.value_counts().sort_index())

print("\nPercentage distribution:")
print(
    (skill_set_sizes.value_counts(normalize=True)
     .sort_index() * 100).round(2)
)

Skill count distribution:
skill_set
0     11163
1    136536
2     25677
3      2885
4      1063
Name: count, dtype: int64

Percentage distribution:
skill_set
0     6.30
1    77.00
2    14.48
3     1.63
4     0.60
Name: proportion, dtype: float64


In [6]:
sample_data = fold_train_df.head(5000).copy()

sample_features = fe.add_same_skill_history_features(
    sample_data,
    alpha=10,
    prior_data=fold_train_df
)

display(
    sample_features[
        [
            "order_id",
            "skill_id",
            "correct",
            "same_skill_previous_interactions",
            "same_skill_previous_correct",
            "same_skill_global_prior",
            "same_skill_smoothed_accuracy"
        ]
    ].head(15)
)

,order_id,skill_id,correct,same_skill_previous_interactions,same_skill_previous_correct,same_skill_global_prior,same_skill_smoothed_accuracy
0,21617623,2_37_70,0,0.0,0.0,0.533333,0.533333
1,21617632,2_37_70,1,1.0,0.0,0.516129,0.469208
2,21617641,2_37_70,0,2.0,1.0,0.531250,0.526042
3,21617650,2_37_70,0,3.0,1.0,0.515152,0.473193
4,21617659,2_37_70,0,4.0,1.0,0.500000,0.428571
5,21617667,2_37_70,0,5.0,1.0,0.485714,0.390476
6,21617675,2_37_48_77,0,6.0,1.0,0.434783,0.334239
7,21617692,2_48_79,0,7.0,1.0,0.452381,0.324930
8,21617731,2_37_48,1,8.0,1.0,0.384615,0.269231
9,21617749,2_48_79,0,9.0,2.0,0.454545,0.344498


In [7]:
# %%
columns_to_inspect = [
    "user_id",
    "order_id",
    "skill_id",
    "correct",
    "same_skill_previous_interactions",
    "same_skill_previous_correct",
    "same_skill_global_prior",
    "same_skill_smoothed_accuracy"
]

sample_features[columns_to_inspect].head(15)

,user_id,order_id,skill_id,correct,same_skill_previous_interactions,same_skill_previous_correct,same_skill_global_prior,same_skill_smoothed_accuracy
0,14,21617623,2_37_70,0,0.0,0.0,0.533333,0.533333
1,14,21617632,2_37_70,1,1.0,0.0,0.516129,0.469208
2,14,21617641,2_37_70,0,2.0,1.0,0.531250,0.526042
3,14,21617650,2_37_70,0,3.0,1.0,0.515152,0.473193
4,14,21617659,2_37_70,0,4.0,1.0,0.500000,0.428571
5,14,21617667,2_37_70,0,5.0,1.0,0.485714,0.390476
6,14,21617675,2_37_48_77,0,6.0,1.0,0.434783,0.334239
7,14,21617692,2_48_79,0,7.0,1.0,0.452381,0.324930
8,14,21617731,2_37_48,1,8.0,1.0,0.384615,0.269231
9,14,21617749,2_48_79,0,9.0,2.0,0.454545,0.344498


In [8]:
# %%
print("Missing values:")
print(sample_features[
    [
        "same_skill_previous_interactions",
        "same_skill_previous_correct",
        "same_skill_global_prior",
        "same_skill_smoothed_accuracy"
    ]
].isna().sum())

print("\nGlobal prior range:")
print(
    sample_features["same_skill_global_prior"].min(),
    sample_features["same_skill_global_prior"].max()
)

print("\nSmoothed accuracy range:")
print(
    sample_features["same_skill_smoothed_accuracy"].min(),
    sample_features["same_skill_smoothed_accuracy"].max()
)

print("\nInvalid historical counts:")
print(
    (
        sample_features["same_skill_previous_correct"]
        > sample_features["same_skill_previous_interactions"]
    ).sum()
)

Missing values:
same_skill_previous_interactions    0
same_skill_previous_correct         0
same_skill_global_prior             0
same_skill_smoothed_accuracy        0
dtype: int64

Global prior range:
0.0 1.0

Smoothed accuracy range:
0.0 1.0

Invalid historical counts:
0


In [ ]:
# %%
v21_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    print(f"\nProcessing Fold {fold_number}...")

    # Generate training features
    v21_train_df = fe.add_same_skill_history_features(
        fold_train_df,
        alpha=10,
        prior_data=fold_train_df
    )

    # Generate validation features
    v21_val_df = fe.add_same_skill_history_features(
        fold_val_df,
        alpha=10,
        prior_data=fold_train_df
    )

    v21_fold_datasets.append(
        (v21_train_df, v21_val_df)
    )

    print("Training features:", v21_train_df.shape)
    print("Validation features:", v21_val_df.shape)

print("\nV2.1 feature generation completed.")


Processing Fold 1...
Training features: (177324, 16)
Validation features: (42659, 16)

Processing Fold 2...
Training features: (176273, 16)
Validation features: (43710, 16)

Processing Fold 3...
Training features: (170186, 16)
Validation features: (49797, 16)

Processing Fold 4...
Training features: (178311, 16)
Validation features: (41672, 16)

Processing Fold 5...
